# M17 — conditioning audit of the Ridge perturbation bound (train-only)M8 derives a Ridge-solution bound conditional on`epsilon_t = ||A_t^{-1} Delta_t||_2 < 1`, but M7 never measured `epsilon_t` —it measured a randomized *action* of `Delta_t` and said so explicitly. Thisnotebook measures the quantity the bound is actually about, for every task atwidths 10,000 and 20,000, and checks the bound against the relative weighterror M7 already recorded.It is **prediction-free**: no accuracy, no logits, no predictions, no `test.pt`.It **cannot** change the recorded status of M6 or M7.The decisive gate is `bound_dominates_measured_weight_error`. A violation wouldindicate an error in the derivation or in the estimator and must be reported,not tuned away. The contract is frozen in`docs/research/SRQ_GENERALIZATION_M17_PROTOCOL.md`.**Abort condition:** if this has not completed by **2026-09-18**, abandon it andcarry the gap into the rebuttal window. The submission must not wait on it.

In [ ]:
REPO_GIT_URL='https://github.com/ZaPhat206/SOHO-CL.git'
REPO_COMMIT='c822bfe77fee862c179993da7e68a838775d298c'
WORK_DIR='/content/SOHO-CL'
FEATURE_CACHE_DIR='/content/srq_m17_cifar_features'
OUTPUT_DIR='/content/srq_m17_output'
M6_NAME='srq_generalization_m6_width_sweep_train_only.zip'
M6_SHA='b2739b9da023ebd2eedb6fdfe01c394e94f252773e847533b35350021c3d239e'
M7_NAME='srq_generalization_m7_error_trajectory_train_only.zip'
M7_SHA='df92adadce046c53efa5b9fcf01435d1fab2a4c71a690b10c3d7c221205acf36'
CONFIG='configs/srq_generalization_m17_conditioning_audit.json'
RUNNER='tools/srq_generalization_m17.py'
AUDIT_DTYPE='float64'
CHECKPOINT_SHA='32aa17d6e17b43500f531d5f6dc9bc93e56ed8841b8a75682e1bb295d722405b'
CHECKPOINT_SIZE=346284714
BATCH_SIZE=128
NUM_WORKERS=2
assert REPO_COMMIT!='REPLACE_WITH_COMMIT_THAT_CONTAINS_M17','Pin the commit that contains the M17 files.'

In [ ]:
# Exact checkout, dependencies, GPU, and canonical-LF source verification.
import hashlib,json,os,shutil,subprocess,sys
from pathlib import Path
os.chdir('/content')
repo=Path(WORK_DIR)
if repo.exists(): shutil.rmtree(repo)
subprocess.run(['git','clone',REPO_GIT_URL,WORK_DIR],check=True)
subprocess.run(['git','checkout','--detach',REPO_COMMIT],cwd=WORK_DIR,check=True)
os.chdir(WORK_DIR)
subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements-kaggle.txt','kagglehub','huggingface_hub'],check=True)
import torch
assert torch.cuda.is_available(),'Select Runtime -> Change runtime type -> GPU.'
def sha_raw(path): return hashlib.sha256(Path(path).read_bytes()).hexdigest()
def sha_source(path): return hashlib.sha256(Path(path).read_bytes().replace(b'\r\n',b'\n')).hexdigest()
EXPECTED={
 'configs/srq_generalization_m17_conditioning_audit.json':'e6f669b57ee2ed8f9915871f4b7e48b3e66f1970051813f771ebc69bf547ca90',
 'tools/srq_generalization_m17.py':'c24458e88ecdcdb6a57a34734c16de8646d7d30af933a7bb2b5272f8d07b29a3',
 'tests/test_srq_generalization_m17.py':'e2984b894badc85d85e1656038cbefb73baa86ebd94fad67eff4770db41dc477',
 'docs/research/SRQ_GENERALIZATION_M17_PROTOCOL.md':'2b031b9172b86c5ad4622850670af7b0839aec9070411d94afb40cc316b40f28',
 'methods/analytic_ridge/backends.py':'cb97a6b65991e41af5f52302bcbdeac5ded6dc95774055c64c6eecdbbfb50ad3',
 'tools/experiment_runner.py':'b2c953eea312a98ce4146757fb46a7dd0e4ebe20aa139da59464921b11f8310c',
 'models/backbone.py':'941e449dc6e66ca4018fb0d3ab3218d97ec97f498b557ed220c8332e75850a46',
 'utils/data_utils.py':'3cf85993e231b068ad5ae2f96be608b2e50e9c52f98fb2387fd3badfb44b6764',
 'utils/train_utils.py':'e24983bd3042ad82ec069916ba2853cf1c818cb2911ce193710c8ccd70e86bda'}
for path,expected in EXPECTED.items(): assert sha_source(path)==expected,(path,sha_source(path),expected)
assert subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip()==REPO_COMMIT
assert not subprocess.check_output(['git','status','--porcelain'],text=True).strip()
print('GPU:',torch.cuda.get_device_name(0),'| M17 SOURCE LOCK: PASS')

In [ ]:
# Estimator validation before any GPU time is spent. These tests check the
# power iteration against dense ground truth; if they fail, the run is wasted.
command=[sys.executable,'-B','-m','pytest','-q','-p','no:cacheprovider','tests/test_srq_generalization_m17.py','tests/test_analytic_ridge_backend.py']
completed=subprocess.run(command)
assert completed.returncode==0,'M17 local gates failed; return the complete traceback.'
assert not subprocess.check_output(['git','status','--porcelain'],text=True).strip()
print('M17 ESTIMATOR GATES: PASS')

In [ ]:
# Upload both immutable upstream artifacts. M17 is locked to each of them.
from google.colab import files
os.chdir('/content')
uploaded=files.upload()
assert set(uploaded)=={M6_NAME,M7_NAME},f'Upload exactly {M6_NAME} and {M7_NAME}; got {sorted(uploaded)}'
M6_ARTIFACT=str((Path('/content')/M6_NAME).resolve())
M7_ARTIFACT=str((Path('/content')/M7_NAME).resolve())
assert sha_raw(M6_ARTIFACT)==M6_SHA,(sha_raw(M6_ARTIFACT),M6_SHA)
assert sha_raw(M7_ARTIFACT)==M7_SHA,(sha_raw(M7_ARTIFACT),M7_SHA)
os.chdir(WORK_DIR)
assert not subprocess.check_output(['git','status','--porcelain'],text=True).strip()
print('SOURCE ARTIFACTS VERIFIED')

In [ ]:
# Download the locked checkpoint and processed CIFAR-100 source.
import kagglehub
from huggingface_hub import hf_hub_download
CHECKPOINT_PATH=hf_hub_download(repo_id='timm/vit_base_patch16_224.augreg2_in21k_ft_in1k',filename='model.safetensors')
assert Path(CHECKPOINT_PATH).stat().st_size==CHECKPOINT_SIZE
assert sha_raw(CHECKPOINT_PATH)==CHECKPOINT_SHA
CIFAR_ROOT=kagglehub.dataset_download('zaphat206/cifar-100')
print('CHECKPOINT:',CHECKPOINT_PATH)
print('CIFAR ROOT:',CIFAR_ROOT)

In [ ]:
# Materialize TRAIN features only; held-out features are forbidden.
cache=Path(FEATURE_CACHE_DIR)
if not (cache/'train.pt').is_file():
    command=[sys.executable,'-u','tools/experiment_runner.py','--extract-features-only','--extract-train-only','--root',CIFAR_ROOT,'--backbone-checkpoint',CHECKPOINT_PATH,'--backbone-checkpoint-size',str(CHECKPOINT_SIZE),'--backbone-checkpoint-sha256',CHECKPOINT_SHA,'--feature-cache-dir',FEATURE_CACHE_DIR,'--output-dir','/content/unused_m17','--dataset','CIFAR-100','--model-name','vit_base_patch16_224','--data-augmentation','vit','--seed','2025','--num-classes','100','--num-tasks','10','--device','cuda','--batch-size',str(BATCH_SIZE),'--num-workers',str(NUM_WORKERS)]
    subprocess.run(command,check=True)
assert (cache/'train.pt').is_file() and not (cache/'test.pt').exists()
metadata=json.loads((cache/'metadata.json').read_text())
assert metadata['feature_dim']==768 and metadata['finite'] is True
assert sha_raw(cache/'train.pt')=='ba53b82123964708123fe868a69d688bfdda4d66b1a3b7ea56329cc9ee9321dc'
print('TRAIN CACHE READY:',metadata.get('train_shape'),'| test.pt absent')

### Memory note — read before running on a T4`A_t` is never held densely: it is factored once per task and every laterproduct is taken as `L (L^T v)`. At width 20,000 in `float64` the peak is thenabout **9.6 GB** (Cholesky factor 3.2, decoded factor 3.2, plus the float32 Gramthe backend carries). A 16 GB T4 fits that, but without much room.If the run cell dies with an out-of-memory error at width 20,000:1. rerun with `AUDIT_DTYPE='float32'` — do **not** treat that as equivalent.   float32 raises the noise floor on `epsilon_t` from about `2e-16 * kappa` to   `1.2e-7 * kappa`. The run gates on `epsilon_resolved_above_noise_floor`, so   it will tell you whether the measurement survived rather than quietly   reporting rounding error;2. or switch the runtime to a larger GPU and keep `float64`.TF32 is disabled explicitly by the runner regardless of card, and the lockedprecision settings and device name are recorded in the artifact provenance.On a T4 this changes nothing — Turing has no TF32 path — but it is what makesthe result portable if the audit is ever repeated on Ampere.

In [ ]:
# 2 widths x 10 tasks. Prediction-free: no accuracy is computed anywhere.
command=[sys.executable,'-u',RUNNER,'--config',CONFIG,'--feature-cache-dir',FEATURE_CACHE_DIR,'--output-dir',OUTPUT_DIR,'--m6-artifact',M6_ARTIFACT,'--m7-artifact',M7_ARTIFACT,'--device','cuda','--audit-dtype',AUDIT_DTYPE,'--require-clean-git']
print('M17 START: widths 10000 and 20000, ten tasks each, no test data.',flush=True)
completed=subprocess.run(command)
assert completed.returncode==0,'M17 failed; return the complete traceback.'

In [ ]:
# Gate report and the conditioning table. No predictive metric appears here.
results=json.loads((Path(OUTPUT_DIR)/'m17_results.json').read_text())
print('STATUS:',results['status'])
for name,value in results['gates'].items():
    print(f"  {'PASS' if value else 'FAIL'}  {name}")
print()
print('maximum epsilon        :',results['summary']['maximum_epsilon'])
print('max Cholesky residual  :',results['summary']['maximum_cholesky_relative_residual'])
for width,block in results['summary']['per_width'].items():
    print(f"\n--- width {width} ---")
    for key,value in block.items(): print(f'  {key:34} {value}')

In [ ]:
# Per-task view: does the derived bound stay above the measured weight error?
import csv
rows=list(csv.DictReader(open(Path(OUTPUT_DIR)/'conditioning_audit.csv')))
print(f"{'w':>6} {'t':>3} {'kappa(A_t)':>13} {'epsilon':>12} {'bound':>12} {'M7 weight err':>14} {'bound/meas':>11}")
for r in rows:
    meas=r['m7_relative_weight_error']
    ratio=r['bound_over_measured']
    print(f"{int(r['width']):>6} {int(r['task']):>3} {float(r['condition_number']):>13.4e} {float(r['epsilon']):>12.5e} {float(r['ridge_perturbation_bound']):>12.5e} {float(meas):>14.6f} {float(ratio):>11.2f}")

In [ ]:
# Package the artifact with its hash, then download it.
import shutil
archive=shutil.make_archive('/content/srq_generalization_m17_conditioning_audit','zip',OUTPUT_DIR)
print('ARTIFACT:',archive)
print('SHA-256 :',sha_raw(archive))
print('results SHA-256:',sha_raw(Path(OUTPUT_DIR)/'m17_results.json'))
from google.colab import files as _files
_files.download(archive)